Kopiujemy potrzebne pliki, u nas bierzemy do porównania
- control 
- sucrose
- torin1
- oraz inputy

Blackliste bęzdiemy porównywać sobie potem. 

In [ ]:
cp -v $DIR/Sucrose_rep* $DIR/Torin1*  $DIR/Control_rep* $DIR/Input*  base_files/

In [ ]:
%%bash
OUT_DIR=~/labs5_all/idr_files # add path to the output folder
mkdir -p $OUT_DIR

# Loop over conditions (CCCP, Control, Sucrose, Torin1)
for CONDITION in CCCP Control Sucrose Torin1; do
    REP1=${PEAK_DIR}/${CONDITION}_rep1.sorted.blacklist.mapq20_peaks.narrowPeak
    REP2=${PEAK_DIR}/${CONDITION}_rep2.sorted.blacklist.mapq20_peaks.narrowPeak

    if [[ -f $REP1 && -f $REP2 ]]; then
        echo "Running IDR for $CONDITION ..."
        idr \
            --samples $REP1 $REP2 \
            --rank p.value \
            --input-file-type narrowPeak \
            --output-file ${OUT_DIR}/${CONDITION}_idr.narrowPeak \
            --idr-threshold 0.05 \
            --plot \
            --log-output-file ${OUT_DIR}/${CONDITION}_idr.log
    else
        echo "Missing replicates for $CONDITION, skipping."
    fi

done

In [ ]:
# Loop over BAM files
# =========================
for bam in $BAM_DIR/*.bam; do
    base=$(basename $bam .bam)

    # Skip input BAM itself
    if [[ "$(basename "$bam")" == "$(basename "$INPUT")" ]]; then
        continue
    fi

    echo "Calling peaks for $bam using MACS2..."

    macs2 callpeak \
        -t $bam \
        -c $INPUT \
        -f BAM \
        -g 2.7e9 \
        -n $base \
        --outdir $PEAKS_DIR \
        -p 0.01 \
        2> $PEAKS_DIR/${base}_macs2.log

done

echo "All MACS2 peak calling done. Results are in $PEAKS_DIR"